# Pickup Requests Map

Plots data from `full_test_data.json`:

- **Requested pickup (home)** — every employee's requested pickup location (their home address), red
- **Assigned pickup stops** — each car's pickup stops from `vehicle_pickup_locations`, blue
- **Car parking** — each car's parking location, green (car icon)
- **Office** — black

Filterable by **shift**, **zone**, and **car** (dropdowns). Selecting a **car** shows:

- that car's **parking location**
- that car's **assigned pickup stops**
- **the home addresses of the employees assigned to that car**

Selecting a **zone** (with car = "All cars") filters the requests to that zone.


## Setup

In [25]:
# %pip install -q folium ipywidgets


## Load data

In [26]:
import json
from datetime import datetime

import folium

try:
    from IPython.display import display, clear_output
except Exception:
    def display(*args, **kwargs):
        for item in args:
            print(item)
    def clear_output(*args, **kwargs):
        return None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

DATA_PATH = "full_test_data.json"  # place this file in the same folder as the notebook
OFFICE = (23.770204034678137, 90.40845882507914)
OFFICE_NAME = "Augmedix Bangladesh"

with open(DATA_PATH) as f:
    _data = json.load(f)

USERS = _data["users"]
EMPLOYEES = _data["employees"]
VEHICLES = _data["vehicles"]
VEHICLE_PICKUP_LOCATIONS = _data["vehicle_pickup_locations"]
PICKUP_REQUESTS = _data["pickup_requests"]
DROPOFF_REQUESTS = _data["dropoff_requests"]

print(f"Users: {len(USERS)} | Employees: {len(EMPLOYEES)} | Vehicles: {len(VEHICLES)}")
print(f"Vehicle pickup locations: {len(VEHICLE_PICKUP_LOCATIONS)} | Pickup requests: {len(PICKUP_REQUESTS)} | Dropoff requests: {len(DROPOFF_REQUESTS)}")


Users: 461 | Employees: 414 | Vehicles: 47
Vehicle pickup locations: 186 | Pickup requests: 414 | Dropoff requests: 414


## Build records, stops, and a vehicle lookup

In [27]:
user_by_email = {u["email"]: u for u in USERS}
vehicle_by_plate = {v["plate_no"]: v for v in VEHICLES}

def employee_name(email):
    u = user_by_email.get(email)
    return u["name"] if u else email

def shift_label(shift_time):
    try:
        t = datetime.strptime(shift_time, "%H:%M:%S")
        return t.strftime("%I:%M %p").lstrip("0")
    except (ValueError, TypeError):
        return str(shift_time)

def car_label(plate):
    if not plate:
        return "Unassigned"
    v = vehicle_by_plate.get(plate)
    if v:
        return f"{plate} ({v.get('capacity')} seats)"
    return plate

# requested pickup = home address
records = []
for pr in PICKUP_REQUESTS:
    records.append({
        "employee_email": pr.get("employee_email"),
        "employee_name": employee_name(pr.get("employee_email")),
        "zone_name": pr.get("zone_name") or "Unknown",
        "vehicle_plate": pr.get("vehicle_plate"),
        "shift_time": pr.get("shift_start_time"),
        "lat": pr.get("pickup_lat"),
        "lng": pr.get("pickup_lng"),
        "request_type": pr.get("request_type"),
        "status": pr.get("status"),
    })

# assigned pickup stops per car
stops = []
for s in VEHICLE_PICKUP_LOCATIONS:
    stops.append({
        "vehicle_plate": s.get("vehicle_plate"),
        "location_name": s.get("location_name"),
        "lat": s.get("pickup_lat"),
        "lng": s.get("pickup_lng"),
        "shift_time": s.get("shift_time"),
        "sequence_order": s.get("sequence_order"),
    })

# cars with their zone + parking coords (for the parking markers / zone lookup)
cars = []
for v in VEHICLES:
    cars.append({
        "plate_no": v.get("plate_no"),
        "zone_name": v.get("zone_name"),
        "parking_lat": v.get("parking_lat"),
        "parking_lng": v.get("parking_lng"),
        "capacity": v.get("capacity"),
        "driver_email": v.get("driver_email"),
    })

car_zone = {c["plate_no"]: c["zone_name"] for c in cars}

distinct_shifts = sorted({r["shift_time"] for r in records if r["shift_time"]})
distinct_zones = sorted({r["zone_name"] for r in records})
distinct_cars = sorted(c["plate_no"] for c in cars if c["plate_no"])

print(f"Records built: {len(records)}")
print(f"Assigned pickup stops: {len(stops)}")
print(f"Cars: {len(cars)}")
print(f"Shifts found: {[shift_label(s) for s in distinct_shifts]}")
print(f"Zones found: {distinct_zones}")


Records built: 414
Assigned pickup stops: 186
Cars: 47
Shifts found: ['12:00 AM', '1:00 AM', '2:00 AM', '3:00 AM', '4:00 AM', '5:00 AM', '6:00 AM', '10:00 PM', '11:00 PM']
Zones found: ['Zone-1', 'Zone-2', 'Zone-3']


## Statistics

Counts of pickup/dropoff requests per zone & shift, and a per-car summary (plate, zone, parking, stop count).


In [28]:
from collections import defaultdict

# 1) Pickup & dropoff requests per zone and per shift/drop time
pickup_counts = defaultdict(int)
for pr in PICKUP_REQUESTS:
    pickup_counts[(pr.get("zone_name") or "Unknown", pr.get("shift_start_time"))] += 1

dropoff_counts = defaultdict(int)
for dr in DROPOFF_REQUESTS:
    dropoff_counts[(dr.get("zone_name") or "Unknown", dr.get("drop_time"))] += 1

print("=== 1a) Pickup requests per zone & shift (shift_start_time) ===")
for z in distinct_zones:
    pairs = sorted((s, c) for (zz, s), c in pickup_counts.items() if zz == z)
    for s, c in pairs:
        print(f"    {z:8s} {shift_label(s):>10s} : {c} pickup")

print()
print("=== 1b) Dropoff requests per zone & drop time ===")
for z in distinct_zones:
    pairs = sorted((s, c) for (zz, s), c in dropoff_counts.items() if zz == z)
    for s, c in pairs:
        print(f"    {z:8s} {shift_label(s):>10s} : {c} dropoff")

print()
print("Totals per zone:")
for z in distinct_zones:
    p = sum(c for (zz, s), c in pickup_counts.items() if zz == z)
    d = sum(c for (zz, s), c in dropoff_counts.items() if zz == z)
    print(f"    {z:8s} : {p} pickups, {d} dropoffs")

print()
print("=== 2) Cars: plate, zone, parking, stops ===")
stops_per_car = defaultdict(int)
for s in VEHICLE_PICKUP_LOCATIONS:
    stops_per_car[s.get("vehicle_plate")] += 1

print(f"Total cars: {len(VEHICLES)}")
print(f"{'plate_no':32s} {'zone':8s} {'capacity':>8s} {'stops':>5s}  parking")
for v in sorted(VEHICLES, key=lambda x: ((x.get('zone_name') or ''), (x.get('plate_no') or ''))):
    plate = v.get("plate_no")
    has_p = v.get("parking_lat") is not None and v.get("parking_lng") is not None
    parking = f"({v.get('parking_lat')}, {v.get('parking_lng')})" if has_p else "None"
    print(f"{plate:32s} {v.get('zone_name') or '?':8s} {v.get('capacity'):>8d} {stops_per_car.get(plate, 0):>5d}  {parking}")


=== 1a) Pickup requests per zone & shift (shift_start_time) ===
    Zone-1     12:00 AM : 10 pickup
    Zone-1      1:00 AM : 8 pickup
    Zone-1      2:00 AM : 5 pickup
    Zone-1      3:00 AM : 2 pickup
    Zone-1      5:00 AM : 4 pickup
    Zone-1      6:00 AM : 3 pickup
    Zone-1     10:00 PM : 148 pickup
    Zone-1     11:00 PM : 38 pickup
    Zone-2     12:00 AM : 1 pickup
    Zone-2      1:00 AM : 4 pickup
    Zone-2      2:00 AM : 2 pickup
    Zone-2      3:00 AM : 1 pickup
    Zone-2      4:00 AM : 1 pickup
    Zone-2      5:00 AM : 2 pickup
    Zone-2      6:00 AM : 3 pickup
    Zone-2     10:00 PM : 64 pickup
    Zone-2     11:00 PM : 17 pickup
    Zone-3     12:00 AM : 9 pickup
    Zone-3      1:00 AM : 2 pickup
    Zone-3      2:00 AM : 1 pickup
    Zone-3      3:00 AM : 1 pickup
    Zone-3      5:00 AM : 1 pickup
    Zone-3      6:00 AM : 3 pickup
    Zone-3     10:00 PM : 65 pickup
    Zone-3     11:00 PM : 19 pickup

=== 1b) Dropoff requests per zone & drop time ===
  

## Map builder

In [29]:
HOME_COLOR = "#dc2626"     # red -- requested pickup (home address)
STOP_COLOR = "#2563eb"    # blue -- assigned pickup stop (bus stop)
OFFICE_COLOR = "#000000"  # black -- office

def _parking_marker(car, layer):
    if car["parking_lat"] is None or car["parking_lng"] is None:
        return
    folium.Marker(
        (car["parking_lat"], car["parking_lng"]),
        tooltip=f"Parking - {car_label(car['plate_no'])}",
        popup=folium.Popup(
            f"<b>Car parking</b><br>Car: {car_label(car['plate_no'])}<br>"
            f"Zone: {car['zone_name']}<br>Driver: {car['driver_email']}",
            max_width=280,
        ),
        icon=folium.Icon(color="green", icon="car", prefix="fa"),
    ).add_to(layer)

def build_map(selected_shift="All", selected_zone="All", selected_car="All"):
    m = folium.Map(
        location=OFFICE,
        zoom_start=12,
        tiles="OpenStreetMap",
    )

    folium.Marker(
        OFFICE,
        tooltip=OFFICE_NAME,
        popup=f"<b>{OFFICE_NAME}</b><br>Office",
        icon=folium.Icon(color="black", icon="briefcase", prefix="fa"),
    ).add_to(m)

    home_layer = folium.FeatureGroup(name="Requested pickup (home)", show=True)
    stop_layer = folium.FeatureGroup(name="Assigned pickup stops", show=True)
    parking_layer = folium.FeatureGroup(name="Car parking", show=True)

    if selected_car != "All":
        # Selected car: show its parking, its stops, and the addresses of the
        # employees assigned to that car (their requested pickup = home).
        subset = [r for r in records if r["vehicle_plate"] == selected_car]
        if selected_shift != "All":
            subset = [r for r in subset if r["shift_time"] == selected_shift]
        stops_subset = [s for s in stops if s["vehicle_plate"] == selected_car]
        if selected_shift != "All":
            stops_subset = [s for s in stops_subset if s["shift_time"] == selected_shift]
        parking_subset = [c for c in cars if c["plate_no"] == selected_car]
    else:
        # No car selected: shift + zone filters apply to requests; all cars' stops + parking.
        subset = records
        if selected_shift != "All":
            subset = [r for r in subset if r["shift_time"] == selected_shift]
        if selected_zone != "All":
            subset = [r for r in subset if r["zone_name"] == selected_zone]
        stops_subset = stops
        if selected_shift != "All":
            stops_subset = [s for s in stops_subset if s["shift_time"] == selected_shift]
        parking_subset = cars
        if selected_zone != "All":
            parking_subset = [c for c in parking_subset if c["zone_name"] == selected_zone]

    for r in subset:
        if r["lat"] is None or r["lng"] is None:
            continue
        label = f"{r['employee_name']} | {r['zone_name']} | {shift_label(r['shift_time'])}"
        folium.CircleMarker(
            location=(r["lat"], r["lng"]),
            radius=5, color=HOME_COLOR, fill=True, fill_color=HOME_COLOR, fill_opacity=0.85,
            tooltip=label,
            popup=folium.Popup(
                f"<b>Requested pickup (home)</b><br>Name: {r['employee_name']}<br>"
                f"Zone: {r['zone_name']}<br>Shift: {shift_label(r['shift_time'])}<br>"
                f"Car: {car_label(r['vehicle_plate'])}<br>Type: {r['request_type']}<br>"
                f"Status: {r['status']}",
                max_width=280,
            ),
        ).add_to(home_layer)

    for s in stops_subset:
        if s["lat"] is None or s["lng"] is None:
            continue
        folium.CircleMarker(
            location=(s["lat"], s["lng"]),
            radius=6, color=STOP_COLOR, fill=True, fill_color=STOP_COLOR, fill_opacity=0.9,
            tooltip=f"Stop - {s['location_name']}",
            popup=folium.Popup(
                f"<b>Assigned pickup stop</b><br>{s['location_name']}<br>"
                f"Car: {car_label(s['vehicle_plate'])}<br>Shift: {shift_label(s['shift_time'])}<br>"
                f"Sequence: {s['sequence_order']}",
                max_width=280,
            ),
        ).add_to(stop_layer)

    for c in parking_subset:
        _parking_marker(c, parking_layer)

    home_layer.add_to(m)
    stop_layer.add_to(m)
    parking_layer.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    legend_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white;
        padding: 10px 14px; border: 1px solid #ccc; border-radius: 6px;
        box-shadow: 0 1px 4px rgba(0,0,0,0.3); font-size: 13px; line-height: 1.6;">
      <b>Legend</b><br>
      <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{HOME_COLOR};margin-right:6px;"></span>Requested pickup (home)<br>
      <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{STOP_COLOR};margin-right:6px;"></span>Assigned pickup stop<br>
      <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:green;margin-right:6px;"></span>Car parking<br>
      <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{OFFICE_COLOR};margin-right:6px;"></span>{OFFICE_NAME}
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    return m


## Interactive view — filter by shift, zone, and car

In [30]:
def show_selector():
    if widgets is None:
        print("ipywidgets not available -- showing all requests with no filter.")
        m = build_map()
        m.save("pickup_requests_map.html")
        display(m)
        return

    shift_dropdown = widgets.Dropdown(
        options=[("All shifts", "All")] + [(shift_label(s), s) for s in distinct_shifts],
        value="All", description="Shift:",
    )
    zone_dropdown = widgets.Dropdown(
        options=[("All zones", "All")] + [(z, z) for z in distinct_zones],
        value="All", description="Zone:",
    )
    car_dropdown = widgets.Dropdown(
        options=[("All cars", "All")] + [(car_label(c), c) for c in distinct_cars],
        value="All", description="Car:",
    )
    output = widgets.Output()

    def redraw(*_):
        with output:
            clear_output(wait=True)
            m = build_map(
                selected_shift=shift_dropdown.value,
                selected_zone=zone_dropdown.value,
                selected_car=car_dropdown.value,
            )
            m.save("pickup_requests_map.html")
            display(m)

    shift_dropdown.observe(redraw, names="value")
    zone_dropdown.observe(redraw, names="value")
    car_dropdown.observe(redraw, names="value")
    redraw()
    display(widgets.VBox([widgets.HBox([shift_dropdown, zone_dropdown, car_dropdown]), output]))

show_selector()
print(f"\n{len(records)} pickup requests + {len(stops)} assigned stops + {len(cars)} parking spots plotted (filtered live).")
print("Also saved to pickup_requests_map.html for the current filter selection.")



414 pickup requests + 186 assigned stops + 47 parking spots plotted (filtered live).
Also saved to pickup_requests_map.html for the current filter selection.


## OSRM routing demo (10 PM shift)

Queries the local OSRM server (`localhost:5000`) for a pickup sequence and shows:

- **Route distance** and **route time** (OSRM driving duration)
- **Required time** (the 8 PM -> 10 PM window = 2 h)
- **Slack** (is the route feasible within the required time?)
- The route drawn on a map (purple line), pickup stops (blue) + office (black)

Edit `PICKUP_SEQUENCE` below to try any other car/shift.


In [31]:
import urllib.request, json
import folium

# Pickup sequence for the 10 PM shift (lat, lng) -- the stops in order.
# Edit this list to try any other car/shift.
PICKUP_SEQUENCE = [
    (23.80714656742061, 90.35633672528908),
    (23.82788200168664, 90.36354011890172),
    (23.83808851790453, 90.3713507108436),
    (23.822935478012823, 90.39340919577835),
]

# Office (Augmedix Bangladesh) -- the final destination of the trip.
OFFICE = (23.770204034678137, 90.40845882507914)

# Full route: pickup stops in sequence, then the office.
ROUTE_POINTS = PICKUP_SEQUENCE + [OFFICE]

OSRM_URL = (
    "http://localhost:5000/route/v1/driving/"
    + ";".join(f"{lng},{lat}" for lat, lng in ROUTE_POINTS)
    + "?overview=full&geometries=geojson&steps=true"
)

print("Querying OSRM at localhost:5000 ...")
try:
    with urllib.request.urlopen(OSRM_URL, timeout=10) as resp:
        data = json.load(resp)
    if data.get("code") != "Ok":
        print("OSRM error:", data.get("message", data.get("code")))
    else:
        route = data["routes"][0]
        distance_km = route["distance"] / 1000
        duration_min = route["duration"] / 60
        coords = route["geometry"]["coordinates"]  # [[lng, lat], ...]
        polyline = [(lat, lng) for lng, lat in coords]

        # Required time: 8 PM -> 10 PM window = 2 hours
        REQUIRED_MIN = 120

        print()
        print("=== OSRM route for 10 PM shift ===")
        print(f"Stops (pickups + office): {len(ROUTE_POINTS)}")
        print(f"Route distance : {distance_km:.2f} km")
        print(f"Route time     : {duration_min:.1f} min ({duration_min/60:.2f} h)")
        print(f"Required time : {REQUIRED_MIN} min (8 PM -> 10 PM window)")
        slack = REQUIRED_MIN - duration_min
        print(f"Slack          : {slack:.1f} min "
              f"({'OK - feasible' if slack >= 0 else 'OVER BUDGET'})")
        print()

        # Draw the route
        m = folium.Map(location=PICKUP_SEQUENCE[0], zoom_start=13,
                       tiles="OpenStreetMap")
        folium.PolyLine(polyline, color="purple", weight=5,
                            opacity=0.8).add_to(m)
        for i, (lat, lng) in enumerate(PICKUP_SEQUENCE, 1):
            folium.CircleMarker(
                (lat, lng), radius=8, color="blue", fill=True,
                fill_opacity=0.9,
                tooltip=f"Pickup {i}: {lat:.5f}, {lng:.5f}",
            ).add_to(m)
        folium.Marker(
            OFFICE, tooltip="Office (Augmedix Bangladesh)",
            icon=folium.Icon(color="black", icon="briefcase", prefix="fa"),
        ).add_to(m)
        m.save("osrm_route_demo.html")
        display(m)
except urllib.error.URLError as e:
    print()
    print("OSRM server not reachable at localhost:5000.")
    print("Reason:", e)
    print("Start the OSRM server (the one showing 'Listening on 0.0.0.0:5000'),")
    print("then re-run this cell.")


Querying OSRM at localhost:5000 ...

=== OSRM route for 10 PM shift ===
Stops (pickups + office): 5
Route distance : 18.33 km
Route time     : 28.5 min (0.48 h)
Required time : 120 min (8 PM -> 10 PM window)
Slack          : 91.5 min (OK - feasible)



## Traffic-aware route time (trained model, not just OSRM free-flow)

The cell above only gives OSRM's **free-flow** driving duration -- no traffic. The cell below
uses the trained ETA model from `../trained-model/inference_bundle.joblib` (the Dhaka GPS-trace
pipeline) to predict the **real, traffic-conditioned** time for the same `PICKUP_SEQUENCE`,
leg by leg, chaining the clock forward so each leg's time-of-day features reflect when it
would actually happen.

Note: the OSRM demo above queries `localhost:5000`, but on this machine the OSRM container is
actually published on **port 5050** (5000 is taken by an unrelated container) -- this cell uses
the port stored in the bundle (`bundle["osrm_base"]`) so it works regardless of what's on 5000.

Caveat worth keeping in mind: the model was trained only on real Dhaka GPS traces, which don't
necessarily cover this shuttle service's specific pickup zones -- if a leg's `od_zone_pair_id`
was never seen in training, this falls back to a coarser time-of-day-only estimate (printed
per leg), and both models (Random Forest, XGBoost) are shown since they can disagree
noticeably on such out-of-distribution legs.

In [ ]:
import os, math
import numpy as np
import pandas as pd
import requests
import joblib

BUNDLE_PATH = os.path.join("..", "trained-model", "inference_bundle.joblib")
bundle = joblib.load(BUNDLE_PATH)
print(f"Loaded inference bundle from {BUNDLE_PATH}")
print(f"OSRM base used by the bundle: {bundle['osrm_base']}")

# ---- same helper functions as training_pipeline.ipynb Step 18 (self-contained here) ----
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlmb = math.radians(lon2 - lon1)
    x = math.sin(dlmb) * math.cos(p2)
    y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dlmb)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def traffic_bucket(h):
    if h < 6: return "night"
    elif h < 8: return "morning_offpeak"
    elif h < 10: return "morning_rush"
    elif h < 17: return "midday"
    elif h < 21: return "evening_rush"
    else: return "evening_winddown"

FIXED_HOLIDAYS_MD = {(2, 21), (3, 26), (4, 14), (5, 1), (8, 15), (12, 16), (12, 25)}

def grid_cell_from_bundle(lat, lon, b):
    gp = b["grid_params"]
    r = min(int((lat - gp["min_lat"]) / gp["lat_step"]), gp["n_rows"] - 1)
    c = min(int((lon - gp["min_lon"]) / gp["lon_step"]), gp["n_cols"] - 1)
    return r * gp["n_cols"] + c

def build_feature_row(src_lat, src_lon, dst_lat, dst_lon, query_time, b=bundle):
    dt = pd.to_datetime(query_time)
    haversine_distance_km = haversine_km(src_lat, src_lon, dst_lat, dst_lon)
    bearing_degrees = bearing_deg(src_lat, src_lon, dst_lat, dst_lon)

    url = f"{b['osrm_base']}/route/v1/driving/{src_lon},{src_lat};{dst_lon},{dst_lat}?overview=false"
    route = requests.get(url, timeout=5).json()["routes"][0]
    osrm_route_distance_km = route["distance"] / 1000.0
    osrm_free_flow_duration_sec = route["duration"]
    route_directness_ratio = (haversine_distance_km / osrm_route_distance_km) if osrm_route_distance_km > 0 else np.nan

    src_zone_id = grid_cell_from_bundle(src_lat, src_lon, b)
    dst_zone_id = grid_cell_from_bundle(dst_lat, dst_lon, b)
    od_zone_pair_id = f"{src_zone_id}_{dst_zone_id}"

    hour = dt.hour + dt.minute / 60.0 + dt.second / 3600.0
    day_of_week = dt.day_name()
    is_friday, is_saturday = day_of_week == "Friday", day_of_week == "Saturday"
    is_weekend = is_friday or is_saturday
    rush_hour_flag = (8 <= hour < 10) or (17 <= hour < 21)
    bucket = traffic_bucket(hour)
    is_holiday = (dt.month, dt.day) in FIXED_HOLIDAYS_MD

    lvl1, lvl2, lvl3 = b["lvl1"], b["lvl2"], b["lvl3"]
    key1 = (od_zone_pair_id, bucket)
    od_pair_seen = od_zone_pair_id in lvl2.index
    if key1 in lvl1.index and lvl1.loc[key1, "n_trips"] >= b["min_support"]:
        historical_avg_speed_kmh, level = lvl1.loc[key1, "speed"], "od_pair+period"
    elif od_pair_seen and lvl2.loc[od_zone_pair_id, "n_trips"] >= b["min_support"]:
        supported = bucket in lvl3.index and lvl3.loc[bucket, "n_trips"] >= b["min_support"]
        factor = lvl3.loc[bucket, "speed"] / b["global_speed"] if supported else 1.0
        historical_avg_speed_kmh, level = lvl2.loc[od_zone_pair_id, "speed"] * factor, "od_pair_only"
    elif bucket in lvl3.index and lvl3.loc[bucket, "n_trips"] >= b["min_support"]:
        historical_avg_speed_kmh, level = lvl3.loc[bucket, "speed"], "period_only"
    else:
        historical_avg_speed_kmh, level = b["global_speed"], "global_fallback"

    row = {
        "src_zone_id": src_zone_id, "dst_zone_id": dst_zone_id, "od_zone_pair_id": od_zone_pair_id,
        "day_of_week": day_of_week, "traffic_period_bucket": bucket,
        "haversine_distance_km": haversine_distance_km, "bearing_degrees": bearing_degrees,
        "osrm_route_distance_km": osrm_route_distance_km,
        "osrm_free_flow_duration_sec": osrm_free_flow_duration_sec,
        "route_directness_ratio": route_directness_ratio,
        "hour_sin": np.sin(2 * np.pi * hour / 24.0), "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "is_friday": int(is_friday), "is_saturday": int(is_saturday), "is_weekend": int(is_weekend),
        "is_holiday": int(is_holiday), "rush_hour_flag": int(rush_hour_flag),
        "historical_avg_speed_kmh": historical_avg_speed_kmh,
    }
    return row, level, od_pair_seen

def predict_travel_time(src_lat, src_lon, dst_lat, dst_lon, query_time, b=bundle):
    row, level, od_pair_seen = build_feature_row(src_lat, src_lon, dst_lat, dst_lon, query_time, b)

    xrow = pd.DataFrame([row])
    for c in b["categorical_cols"]:
        xrow[c] = pd.Categorical([str(row[c])], categories=b["cat_categories"][c])
    for c in ["is_friday", "is_saturday", "is_weekend", "is_holiday", "rush_hour_flag"]:
        xrow[c] = xrow[c].astype(int)
    xrow = xrow[b["feature_cols"]]
    xgb_pred_sec = float(np.exp(b["xgb_model"].predict(xrow)[0]))

    orow = pd.get_dummies(pd.DataFrame([row]), columns=b["categorical_cols"])
    orow = orow.reindex(columns=b["onehot_columns"], fill_value=0)
    rf_pred_sec = float(np.exp(b["rf_model"].predict(orow)[0]))

    return {"xgboost": xgb_pred_sec, "random_forest": rf_pred_sec}, level, od_pair_seen, row

# ---- predict a whole multi-stop sequence, chaining the clock forward leg by leg ----
def predict_sequence_time(points, start_time, chain_on="xgboost", b=bundle):
    """points: list of (lat, lon) in visiting order (e.g. PICKUP_SEQUENCE + [OFFICE]).
    start_time: when the vehicle departs the first point.
    chain_on: which model's prediction advances the clock between legs ("xgboost" or "random_forest")."""
    current_time = pd.to_datetime(start_time)
    total_xgb_sec, total_rf_sec = 0.0, 0.0

    print(f"{'leg':>4}  {'depart':>19}  {'od_pair':>9}  {'level':>15}  {'seen':>5}  "
          f"{'xgboost(s)':>11}  {'random_forest(s)':>16}")
    for i in range(len(points) - 1):
        (src_lat, src_lon), (dst_lat, dst_lon) = points[i], points[i + 1]
        preds, level, seen, row = predict_travel_time(src_lat, src_lon, dst_lat, dst_lon, current_time, b)
        total_xgb_sec += preds["xgboost"]
        total_rf_sec += preds["random_forest"]
        print(f"{i+1:>4}  {str(current_time):>19}  {row['od_zone_pair_id']:>9}  {level:>15}  "
              f"{str(seen):>5}  {preds['xgboost']:>11.1f}  {preds['random_forest']:>16.1f}")
        current_time = current_time + pd.Timedelta(seconds=preds[chain_on])

    return total_xgb_sec, total_rf_sec

# ---- run it on the same PICKUP_SEQUENCE + OFFICE route as the OSRM demo above ----
START_TIME = "2026-09-10T20:00:00"  # edit to try a different departure time

print(f"Predicting traffic-aware time for {len(ROUTE_POINTS)} stops, departing {START_TIME}\n")
total_xgb_sec, total_rf_sec = predict_sequence_time(ROUTE_POINTS, START_TIME)

REQUIRED_MIN = 120  # same 8 PM -> 10 PM window as the OSRM cell above
print()
print(f"=== Trained-model total (traffic-aware) ===")
print(f"XGBoost total       : {total_xgb_sec/60:.1f} min ({total_xgb_sec/3600:.2f} h)  "
      f"slack={REQUIRED_MIN - total_xgb_sec/60:.1f} min "
      f"({'OK - feasible' if REQUIRED_MIN - total_xgb_sec/60 >= 0 else 'OVER BUDGET'})")
print(f"Random Forest total : {total_rf_sec/60:.1f} min ({total_rf_sec/3600:.2f} h)  "
      f"slack={REQUIRED_MIN - total_rf_sec/60:.1f} min "
      f"({'OK - feasible' if REQUIRED_MIN - total_rf_sec/60 >= 0 else 'OVER BUDGET'})")
if "duration_min" in globals():
    print(f"Compare to the OSRM free-flow estimate above: {duration_min:.1f} min -- "
          f"the gap between that and these numbers is exactly the traffic effect this model adds.")
else:
    print("The OSRM-only cell above did not run successfully this time, so there is no")
    print("free-flow number to compare against -- re-run it once OSRM is reachable.")


Loaded inference bundle from ..\trained-model\inference_bundle.joblib
OSRM base used by the bundle: http://localhost:5050
Predicting traffic-aware time for 5 stops, departing 2026-09-10T20:00:00

 leg               depart    od_pair            level   seen   xgboost(s)  random_forest(s)
   1  2026-09-10 20:00:00      19_19     od_pair_only   True        772.7             778.8
   2  2026-09-10 20:12:52.733947753      19_17      period_only  False        465.6             512.1
   3  2026-09-10 20:20:38.299041747       17_8      period_only   True        428.5             349.9
   4  2026-09-10 20:27:46.803131102        8_1      period_only  False       1148.6            1177.1

=== Trained-model total (traffic-aware) ===
XGBoost total       : 46.9 min (0.78 h)  slack=73.1 min (OK - feasible)
Random Forest total : 47.0 min (0.78 h)  slack=73.0 min (OK - feasible)
Compare to the OSRM free-flow estimate above: 28.5 min -- the gap between that and these numbers is exactly the traffic effec